In [ ]:
import pandas as pd
import numpy as np
from mrmr import mrmr_classif
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, confusion_matrix, accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import time
from torch.optim import Adam, AdamW, SGD, RMSprop
import torch.optim.lr_scheduler as lr_scheduler

In [ ]:
# Complex Multi-Head Self-Attention Model
class ComplexMultiHeadSelfAttentionClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=4, num_heads=8, hidden_dim=512, num_layers=3):
        super().__init__()
        
        # Initial feature embedding
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Multiple attention layers
        self.attention_layers = nn.ModuleList([
            nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads, dropout=0.1)
            for _ in range(num_layers)
        ])
        
        # Feed-forward network after attention
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        
        # Final classification layers
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )
        
    def forward(self, x):
        x = x.unsqueeze(0)  # Add sequence dimension
        x = self.embedding(x)
        
        # Apply multiple attention layers
        for attn in self.attention_layers:
            attn_output, _ = attn(x, x, x)
            x = x + attn_output  # Residual connection
            x = self.ffn(x)      # Feed-forward processing
            
        x = x.squeeze(0)
        x = self.classifier(x)
        return x

In [ ]:
# Load and prepare data
data = pd.read_csv('mrmr_selected_1000_features_with_labels.csv')
X = data.drop(['label', 'class'], axis=1)
y = data['label']
class_names = data['class'].unique().tolist()
n_classes = len(class_names)

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to tensors
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train_tensor = torch.FloatTensor(X_train.values).to(device)
X_test_tensor = torch.FloatTensor(X_test.values).to(device)
y_train_tensor = torch.LongTensor(y_train.values).to(device)
y_test_tensor = torch.LongTensor(y_test.values).to(device)

In [ ]:
# Optimizers to compare
optimizers = {
    'Adam': Adam,
    'AdamW': AdamW,
    'SGD': SGD,
    'RMSprop': RMSprop,
    'Adamax': torch.optim.Adamax
}

In [ ]:
# Training and evaluation function
def train_and_evaluate(optimizer_name, optimizer_class):
    model = ComplexMultiHeadSelfAttentionClassifier(
        input_dim=1000,
        num_classes=n_classes,
        num_heads=8,
        hidden_dim=512,
        num_layers=3
    ).to(device)
    
    optimizer = optimizer_class(model.parameters(), lr=0.001)
    scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
    criterion = nn.CrossEntropyLoss()
    
    # Training
    start_time = time.time()
    model.train()
    for epoch in range(30):
        optimizer.zero_grad()
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()
        scheduler.step(loss)
    training_time = time.time() - start_time
    
    # Evaluation
    start_time = time.time()
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        y_pred = torch.argmax(outputs, dim=1)
        y_pred_proba = torch.softmax(outputs, dim=1)
    testing_time = time.time() - start_time
    
    # Convert to numpy
    y_test_np = y_test_tensor.cpu().numpy()
    y_pred_np = y_pred.cpu().numpy()
    y_pred_proba_np = y_pred_proba.cpu().numpy()
    
    # Calculate metrics
    metrics = {
        'optimizer': optimizer_name,
        'accuracy': accuracy_score(y_test_np, y_pred_np),
        'auc': roc_auc_score(y_test_np, y_pred_proba_np, multi_class='ovr'),
        'training_time': training_time,
        'testing_time': testing_time,
        'confusion_matrix': confusion_matrix(y_test_np, y_pred_np).tolist()
    }
    
    return metrics, y_pred_proba_np, model

In [ ]:
# Run experiments
results = []
y_pred_probas = []
method_names = []

for opt_name, opt_class in optimizers.items():
    print(f"\nTraining with {opt_name}...")
    metrics, y_pred_proba, _ = train_and_evaluate(opt_name, opt_class)
    results.append(metrics)
    y_pred_probas.append(y_pred_proba)
    method_names.append(opt_name)

# Save results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv('optimizer_comparison_results.csv', index=False)

In [ ]:
# ROC Curve plotting functions
def plot_roc_curves(y_test, y_pred_probas, class_names, method_names, n_classes):
    plt.figure(figsize=(12, 10), facecolor='white')
    plt.rcParams['font.family'] = 'Times New Roman'
    
    colors = ['blue', 'green', 'red', 'purple', 'orange']
    for m_idx, (method_name, y_pred_proba) in enumerate(zip(method_names, y_pred_probas)):
        for i in range(n_classes):
            y_true_binary = (y_test == i).astype(int)
            fpr, tpr, _ = roc_curve(y_true_binary, y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, color=colors[m_idx], lw=3,
                    label=f'{method_name} - {class_names[i]} (AUC = {roc_auc:.2f})')
    
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=24)
    plt.ylabel('True Positive Rate', fontsize=24)
    plt.title('ROC Curves by Optimizer', fontsize=28)
    plt.legend(loc="lower right", fontsize=18)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    
    for spine in plt.gca().spines.values():
        spine.set_linewidth(2)
        spine.set_edgecolor('black')
    
    plt.savefig('roc_curves_all.png', dpi=1000, bbox_inches='tight')
    plt.savefig('roc_curves_all.pdf', dpi=1000, bbox_inches='tight')
    plt.close()

In [ ]:
def plot_roc_curves_by_class(y_test, y_pred_probas, class_names, method_names, n_classes):
    colors = ['blue', 'green', 'red', 'purple', 'orange']
    
    for i in range(n_classes):
        plt.figure(figsize=(8, 8), facecolor='white')
        plt.rcParams['font.family'] = 'Times New Roman'
        
        for m_idx, (method_name, y_pred_proba) in enumerate(zip(method_names, y_pred_probas)):
            y_true_binary = (y_test == i).astype(int)
            fpr, tpr, _ = roc_curve(y_true_binary, y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, color=colors[m_idx], lw=3,
                    label=f'{method_name} (AUC = {roc_auc:.2f})')
        
        plt.plot([0, 1], [0, 1], 'k--', lw=2)
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate', fontsize=24)
        plt.ylabel('True Positive Rate', fontsize=24)
        plt.title(f'ROC Curve - {class_names[i]}', fontsize=28)
        plt.legend(loc="lower right", fontsize=18)
        plt.xticks(fontsize=18)
        plt.yticks(fontsize=18)
        
        for spine in plt.gca().spines.values():
            spine.set_linewidth(2)
            spine.set_edgecolor('black')
        
        plt.savefig(f'roc_curve_class_{i}.png', dpi=1000, bbox_inches='tight')
        plt.savefig(f'roc_curve_class_{i}.pdf', dpi=1000, bbox_inches='tight')
        plt.close()


In [ ]:
# Generate plots
y_test_np = y_test_tensor.cpu().numpy()
plot_roc_curves(y_test_np, y_pred_probas, class_names, method_names, n_classes)
plot_roc_curves_by_class(y_test_np, y_pred_probas, class_names, method_names, n_classes)

In [ ]:
# Print results
print("\nOptimizer Comparison Results:")
for result in results:
    print(f"\n{result['optimizer']}:")
    print(f"Accuracy: {result['accuracy']:.4f}")
    print(f"AUC: {result['auc']:.4f}")
    print(f"Training Time: {result['training_time']:.2f}s")
    print(f"Testing Time: {result['testing_time']:.2f}s")
    print("Confusion Matrix:")
    print(np.array(result['confusion_matrix']))

print("\nResults saved to 'optimizer_comparison_results.csv'")
print("ROC curves saved as PNG and PDF files")